In [ ]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from utils import extract_url_map
from events import extract_events
from timespan import parse_timespan
from taxonomy import load_taxonomy

In [ ]:
root_path = "../data/schede mappatura/"

skip = {
    # "david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}
    "stern_IS_S_00142": {"rows": 2, "cols": 0},
}

In [ ]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede = [c for c in chrono_schede if not c.endswith(":Zone.Identifier")]
chrono_schede

## A list of individual sources for experimentation

ignored in the oveall logic

In [ ]:
# current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
# current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
# current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"
current = "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx"

chrono_schede = [s for s in chrono_schede if current in s]
chrono_schede

After identification of all sources

# Shortlist processable sources

In [ ]:
overview = {
    "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx": [
        "Josef Stern",
    ],
}

## Put all data together

In [ ]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        name = k.split("/")[0]
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, df.columns)
        df.columns = [
            "event_label",
            "event_type",
            "place_name",
            "place_type",
            "place_category",
            "wikidata_qid",
            "geonames_id",
            "google maps",
            "date_certainty",
            "date_label",
            "memorial_inscription",
            "source_doc",
            "source_timecode",
            "source_quote",
            "external_links",
            "notes",
        ]

        # merge columns 6+ to notes
        df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
            lambda row: " ".join(row.dropna().astype(str)), axis=1
        )
        # df = df.drop(df.columns[6:], axis=1)

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0).astype(str)
df.fillna("", inplace=True)

# Track start/end locations: end_location = current row's place,
# start_location = previous event's place (per person)
start_locations = []
prev_location = {}  # protagonist → last place_name
for _, row in df.iterrows():
    person = row["protagonist"]
    end_loc = row["place_name"].strip()
    start_loc = prev_location.get(person, end_loc)  # fallback to same as end
    start_locations.append(start_loc)
    if end_loc:
        prev_location[person] = end_loc
df["start_location"] = start_locations
df["end_location"] = df["place_name"]

# Collect concepts from event_type, place_type, place_category
concept_labels = set()
for col in ["event_type", "place_type", "place_category"]:
    for val in df[col]:
        v = val.strip()
        if v and v not in ("nan", "None"):
            concept_labels.add(v)
print(f"Concepts to create: {sorted(concept_labels)}")

df


# Locations

In [ ]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df.to_dict(orient="records")):
    # print(row)
    # print(extract_urls(row))
    name = row["place_name"]
    locs[name] = {}
    place_labels = set()
    if "place_type" in row and row["place_type"].strip():
        place_labels |= {row["place_type"].strip()}
    if row["place_category"].strip():
        place_labels |= {row["place_category"].strip()}
    locs[name]["label"] = ",".join(place_labels)

    urls = extract_url_map(row["external_links"])
    if (
        "www.wikidata.org" not in urls
        and "wikidata_id" in row
        and row["wikidata_qid"].strip()
    ):
        locs[name]["www.wikidata.org"] = (
            "https://www.wikidata.org/wiki/" + row["wikidata_qid"].strip()
        )
    if (
        "www.geonames.org" not in urls
        and "geonames_id" in row
        and row["geonames_id"].strip()
    ):
        locs[name]["www.geonames.org"] = (
            "https://www.geonames.org/" + row["geonames_id"].strip().removesuffix(".0")
        )

    locs[name].update(urls[0])
print(locs)

## Add GO concepts as locations

Concepts under 'GO – Luoghi geografici' are geographic locations.
Add them to the locs dict so they get included in locations.xlsx.

In [ ]:
from taxonomy import load_taxonomy

taxonomy = load_taxonomy()

go_concepts = [
    label for label, cat in taxonomy.concept_to_category.items()
    if cat == "GO"
]
# Also include GO sub-category labels
for key, info in taxonomy.sub_categories.items():
    if info.get("parent") == "GO":
        go_concepts.append(info["label"])

for concept_name in go_concepts:
    concept_name = concept_name.strip()
    if not concept_name:
        continue
    if concept_name not in locs:
        locs[concept_name] = {}
    if "label" not in locs[concept_name] or not locs[concept_name]["label"]:
        locs[concept_name]["label"] = "GO"

print(f"Added {len(go_concepts)} GO concepts to locations, total: {len(locs)}")


## Update preexisting locations

In [ ]:
import os
import re
from locations import enrich_locations_xlsx

def _normalize_loc(name):
    """Normalize for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

if os.path.exists("locations.xlsx"):
    rich = pd.read_excel("locations.xlsx", dtype=str)
    rich = rich.set_index(["location"])
else:
    rich = pd.DataFrame()
    rich.index.name = "location"

# Build a normalized index for flexible matching
existing_keys = {_normalize_loc(idx): idx for idx in rich.index}

for location, row in locs.items():
    key = _normalize_loc(location)

    if key in existing_keys:
        # Enrich existing row: only fill absent cells
        real_idx = existing_keys[key]
        if isinstance(row, dict):
            for col, val in row.items():
                if col not in rich.columns:
                    rich[col] = ""
                existing = rich.loc[real_idx, col]
                if isinstance(existing, pd.Series):
                    existing = existing.iloc[0]
                if pd.isna(existing) or str(existing).strip() in ("", "nan"):
                    rich.loc[real_idx, col] = str(val)
    else:
        # New location: add row with provided values
        if isinstance(row, dict):
            for col in row:
                if col not in rich.columns:
                    rich[col] = ""
            rich.loc[location] = {col: str(val) for col, val in row.items()}
        else:
            rich.loc[location] = pd.Series(dtype=str)
        existing_keys[key] = location

rich.to_excel("locations.xlsx")

# Enrich with bag-of-words and super-region columns
enrich_locations_xlsx("locations.xlsx")

# Timespan

In [ ]:
df[["time_start", "time_end"]] = (
    df["date_label"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

# Notes

left unprocessed for now

In [ ]:
set(df["notes"])

# Links

left unprocessed for now

In [ ]:
urls = set()
for cell in df["external_links"]:
    if pd.notna(cell):
        for url in str(cell).split("\n"):
            url = url.strip()
            if url:
                urls.add(url)
urls

# Events

TODO: incomplete due to too much noise. Issues:

- use LL or Lebenslauf, currently extracted as one, but need to be two equivalent
- "alter heimant" instread of "alte heimat"
- "Transport", "Tod des Vaters",  are not label

In [ ]:
taxonomy = load_taxonomy("../data/maxqda/MAXQDA_Code_System.mmd")
events = sorted(set(taxonomy.concept_to_category.keys()), key=lambda x: -len(x))
len(events), events[:15] + ["..."] + events[-15:]
# ",".join(events)

In [ ]:
df["event"] = df["event_label"].apply(lambda x: extract_events(x, events))
df

In [ ]:
from api_client import (
    login, get_or_create_concept,
)
from locations import (
    load_locations_db,
)

login()
load_locations_db()

# === Main import ===

# 1. Create all concepts
print("Creating concepts...")
for label in sorted(concept_labels):
    get_or_create_concept(label)
print(f"  {len(concept_labels)} concept labels processed")